In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Classical feedforward dan control flow (dynamic circuits)

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



# Classical feedforward dan control flow


<details>
<summary><b>Versi paket</b></summary>

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami merekomendasikan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** Versi baru dynamic circuits kini tersedia untuk semua pengguna di semua backend. Kamu sekarang bisa menjalankan dynamic circuits dalam skala utilitas. Lihat [pengumumannya](/announcements/product-updates/2025-09-25-new-dynamic-circuits) untuk detail lebih lanjut.

Dynamic circuits adalah alat yang powerful yang memungkinkan kamu mengukur qubit di tengah eksekusi quantum circuit, lalu melakukan operasi logika klasik di dalam circuit berdasarkan hasil pengukuran mid-circuit tersebut. Proses ini juga dikenal sebagai _classical feedforward_. Meskipun masih tahap awal dalam memahami cara terbaik memanfaatkan dynamic circuits, komunitas riset kuantum sudah mengidentifikasi sejumlah kasus penggunaan, seperti berikut:

* Persiapan quantum state yang efisien, seperti [GHZ state,](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) [W-state,](https://arxiv.org/abs/2403.07604) (untuk informasi lebih lanjut tentang W-state, lihat juga ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)) dan berbagai kelas [matrix product states](https://arxiv.org/abs/2404.16083)
* [Long-range entanglement yang efisien](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) antar qubit di chip yang sama menggunakan shallow circuits
* [Sampling IQP-like circuits](https://arxiv.org/pdf/2505.04705) yang efisien

Namun, peningkatan yang dibawa dynamic circuits ini datang dengan trade-off. Pengukuran mid-circuit dan operasi klasik biasanya memiliki waktu eksekusi yang lebih lama dibanding two-qubit gates, dan peningkatan waktu ini bisa meniadakan keuntungan dari pengurangan kedalaman circuit. Oleh karena itu, pengurangan durasi pengukuran mid-circuit menjadi area fokus perbaikan saat IBM Quantum&reg; merilis [versi baru](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits) dynamic circuits.

[Spesifikasi OpenQASM 3](https://openqasm.com/language/classical.html#looping-and-branching) mendefinisikan sejumlah struktur control-flow, tetapi Qiskit Runtime saat ini hanya mendukung pernyataan kondisional `if`. Di Qiskit SDK, ini sesuai dengan metode [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) pada [QuantumCircuit.](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) Metode ini mengembalikan sebuah [context manager](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) dan biasanya digunakan dalam pernyataan `with`. Panduan ini menjelaskan cara menggunakan pernyataan kondisional tersebut.

> **Note:** Contoh kode dalam panduan ini menggunakan instruksi measure standar untuk pengukuran mid-circuit. Namun, disarankan untuk menggunakan instruksi [`MidCircuitMeasure`](/guides/measure-qubits#midcircuit) sebagai gantinya, jika backend mendukungnya. Lihat [dokumentasi Mid-circuit measurements](/guides/measure-qubits#mid-circuit-measurements) untuk detailnya.
## Pernyataan `if`
Pernyataan `if` digunakan untuk melakukan operasi secara kondisional berdasarkan nilai sebuah classical bit atau register.

Pada contoh di bawah, kita menerapkan Hadamard gate pada sebuah qubit dan mengukurnya. Jika hasilnya adalah 1, maka kita menerapkan X gate pada qubit tersebut, yang efeknya adalah membaliknya kembali ke state 0. Kita kemudian mengukur qubit itu lagi. Hasil pengukuran yang dihasilkan seharusnya 0 dengan probabilitas 100%.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
# Use MidCircuitMeasure() if it's supported by the backend.
# circuit.append(MidCircuitMeasure(), [q0], [c0])
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

Pernyataan `with` bisa diberi target penugasan yang berupa context manager itu sendiri, yang bisa disimpan dan kemudian digunakan untuk membuat blok else, yang dieksekusi setiap kali isi blok `if` *tidak* dieksekusi.

Pada contoh di bawah, kita menginisialisasi register dengan dua qubit dan dua classical bit. Kita menerapkan Hadamard gate pada qubit pertama dan mengukurnya. Jika hasilnya adalah 1, maka kita menerapkan Hadamard gate pada qubit kedua; jika tidak, kita menerapkan X gate pada qubit kedua. Terakhir, kita juga mengukur qubit kedua.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

Selain mengondisikan pada satu classical bit, kamu juga bisa mengondisikan pada nilai sebuah classical register yang terdiri dari beberapa bit.

Pada contoh di bawah, kita menerapkan Hadamard gates pada dua qubit dan mengukurnya. Jika hasilnya adalah `01`, yaitu qubit pertama adalah 1 dan qubit kedua adalah 0, maka kita menerapkan X gate pada qubit ketiga. Terakhir, kita mengukur qubit ketiga. Perlu dicatat bahwa demi kejelasan, kita memilih untuk menentukan state classical bit ketiga, yaitu 0, dalam kondisi `if`. Dalam gambar circuit, kondisi ditunjukkan oleh lingkaran pada classical bit yang dijadikan kondisi. Lingkaran hitam menunjukkan pengondisian pada 1, sedangkan lingkaran putih menunjukkan pengondisian pada 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## Ekspresi klasik
Modul ekspresi klasik Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) berisi representasi eksploratif dari operasi runtime pada nilai klasik selama eksekusi circuit. Karena keterbatasan hardware, hanya kondisi `QuantumCircuit.if_test()` yang saat ini didukung.

Contoh berikut menunjukkan bahwa kamu bisa menggunakan perhitungan paritas untuk membuat n-qubit GHZ state menggunakan dynamic circuits. Pertama, buat $n/2$ pasangan Bell pada qubit yang berdekatan. Kemudian, gabungkan pasangan-pasangan ini menggunakan satu lapisan CNOT gates di antara pasangan. Kamu kemudian mengukur target qubit dari semua CNOT gates sebelumnya dan me-reset setiap qubit yang diukur ke state $\vert 0 \rangle$. Kamu menerapkan $X$ ke setiap situs yang tidak diukur di mana paritas dari semua bit sebelumnya adalah ganjil. Terakhir, CNOT gates diterapkan pada qubit yang diukur untuk memulihkan entanglement yang hilang saat pengukuran.

Dalam perhitungan paritas, elemen pertama dari ekspresi yang dibangun melibatkan pengangkatan objek Python `mr[0]` ke sebuah node [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (`lift` digunakan untuk mengubah objek sembarang menjadi ekspresi klasik). Ini tidak diperlukan untuk `mr[1]` dan kemungkinan classical register berikutnya, karena keduanya adalah input untuk `expr.bit_xor`, dan pengangkatan yang diperlukan dilakukan secara otomatis dalam kasus tersebut. Ekspresi semacam itu bisa dibangun dalam loop dan konstruk lainnya.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [5]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

## Temukan backend yang mendukung dynamic circuits
Untuk menemukan semua backend yang bisa diakses oleh akunmu dan mendukung dynamic circuits, jalankan kode seperti berikut. Contoh ini mengasumsikan bahwa kamu sudah [menyimpan kredensial login.](/guides/save-credentials) Kamu juga bisa [menentukan kredensial secara eksplisit](/guides/initialize-account#explicit) saat menginisialisasi akun layanan Qiskit Runtime. Ini memungkinkan kamu melihat backend yang tersedia pada instansi atau jenis paket tertentu, misalnya.

> **Note:** - Backend yang tersedia untuk akun bergantung pada instansi yang ditentukan dalam kredensial.
> - Versi baru dynamic circuits kini tersedia untuk semua pengguna di semua backend. Lihat [pengumumannya](/announcements/product-updates/2025-09-25-new-dynamic-circuits) untuk detail lebih lanjut.

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
dc_backends = service.backends(dynamic_circuits=True)
print(dc_backends)

[<IBMBackend('ibm_pittsburgh')>, <IBMBackend('ibm_boston')>, <IBMBackend('ibm_fez')>, <IBMBackend('ibm_miami')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_torino')>, <IBMBackend('ibm_kingston')>]


## Qiskit Runtime limitations

Be aware of the following constraints when running dynamic circuits in Qiskit Runtime.

- Due to the limited physical memory on control electronics, there is also a limit on the number of `if` statements and size of their operands. This limit is a function of the number of broadcasts and the number of broadcasted bits in a job (not a circuit).

   When processing an `if` condition, measurement data needs to be transferred to the control logic to make that evaluation. A broadcast is a transfer of unique classical data, and broadcasted bits is the number of classical bits being transferred. Consider the following:

   ```python
   c0 = ClassicalRegister(3)
   c1 = ClassicalRegister(5)
   ...
   with circuit.if_test((c0, 1)) ...
   with circuit.if_test((c0, 3)) ...
   with circuit.if_test((c1[2], 1)) ...
   ```
   In the previous code example, the first two `if_test` objects on `c0` are considered one broadcast because the content of `c0` did not change, and thus does not need to be re-broadcasted. The `if_test` on `c1` is a second broadcast. The first one broadcasts all three bits in `c0` and the second broadcasts just one bit, making a total of four broadcasted bits.

   Currently, if you broadcast 60 bits each time, then the job can have approximately 300 broadcasts. If you broadcast just one bit each time, however, then the job can have 2400 broadcasts.

- The operand used in an `if_test` statement must be 32 or fewer bits. Thus, if you are comparing an entire `ClassicalRegister`, the size of that `ClassicalRegister` must be 32 or fewer bits. If you are comparing just a single bit from a `ClassicalRegister`, however, that `ClassicalRegister` can be of any size (since the operand is only one bit).

   For example, the "Not valid" code block does not work because `cr` is more than 32 bits.  You can, however, use a classical register wider than 32 bits if you are testing only one bit, as shown in the "Valid" code block.

   <Tabs>
   <TabItem value="Not valid" label="Not valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr, 15)):
            ...
      ```
   </TabItem>
   <TabItem value="Valid" label="Valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr[5], 1)):
            ...
      ```
   </TabItem>
   </Tabs>

- Nested conditionals are not allowed. For example, the following code block will not work because it has an `if_test` inside another `if_test`:
   <Tabs>
    <TabItem value="Not valid" label="Not valid">
        ```python
           c1 = ClassicalRegister(1, "c1")
           c2 = ClassicalRegister(2, "c2")
           ...
           with circ.if_test((c1, 1)):
            with circ.if_test(c2, 1)):
             ...
        ```
     </TabItem>
     <TabItem value="Valid" label="Valid">
        ```python
        cr = ClassicalRegister(2)
        ...
        with circuit.if_test((cr, 0b11)):
          ...
        ```
     </TabItem>
    </Tabs>

- Having `reset` or measurements inside conditionals is not supported.
- Arithmetic operations are not supported.
- See the [OpenQASM 3 feature table](/docs/guides/qasm-feature-table) to determine which OpenQASM 3 features are supported on Qiskit and Qiskit Runtime.
- When OpenQASM 3 (instead of `QuantumCircuit`) is used as the input format to pass circuits to Qiskit Runtime primitives, only instructions that can be loaded into Qiskit are supported. Classical operations, for example, are not supported because they cannot be loaded into Qiskit. See [Import an OpenQASM 3 program into Qiskit](/docs/guides/interoperate-qiskit-qasm3#import-an-openqasm-3-program-into-qiskit) for more information.
- The `for`, `while`, and `switch` instructions are not supported.

## Use dynamic circuits with Estimator

Since Estimator does not support dynamic circuits, you can use Sampler and build your own measurement circuits instead. Alternatively, you can use the [Executor primitive,](/docs/guides/directed-execution-model#executor-primitive) which supports dynamic circuits.

To replicate Estimator's behavior, follow this process:

1. Group the terms of all observables into a partition.  This can be done by using the [`PauliList` API,](/docs/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting) for example.
     <Admonition type="note">
      You can use the [`BitArray`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.BitArray#expectation_values) primitive attribute to compute the expectation values of the provided observables.
     </Admonition>
2. Execute one basis change circuit per partition (whichever basis change needs to be done for each partition). See the Measurement bases addon utility  [`measurement_bases` module](https://github.com/Qiskit/qiskit-addon-utils/blob/38ea05431f2e9830adf4ec9265f6d19758a32096/qiskit_addon_utils/exp_vals/measurement_bases.py) for more information. [Get started with utilities.](/docs/guides/qiskit-addons-utils#get-started-with-utilities)
3. Add back together the results for each partition.

## Next steps

<Admonition type="tip" title="Recommendations">
- Learn how to implement accurate dynamic decoupling by using [stretch.](/docs/guides/stretch)
- Learn about the shorter [mid-circuit measurements](/docs/guides/measure-qubits#mid-circuit-measurements) that reduce the circuit time.
- Use [circuit schedule visualization](/docs/guides/visualize-circuit-timing#qiskit-runtime-support) to debug and optimize your dynamic circuits.
</Admonition>